In [1]:
   !pip install -q pyspark

In [5]:

import io, time, zipfile
from contextlib import redirect_stdout
from datetime import date
from pathlib import Path

import pandas as pd
import polars as pl
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

Path("outputs").mkdir(exist_ok=True)
PANEL = "panel_clean.parquet"
REF_POLARS = "bench_polars_lazy.parquet"
DEV_END = "2009-12-31"

# 1. Khoi tao Spark local[*]
t_init0 = time.perf_counter()
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("BFT65B-Part3-PySpark")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
t_init = time.perf_counter() - t_init0

# 2. Xay dung truy van PySpark tuong duong 100% voi 06b_benchmark.py
def build_spark_query():
    df = (
        spark.read.parquet(PANEL)
        .filter(F.col("QDATE") <= F.lit(DEV_END).cast("date"))
        .select("CERT", "QDATE", "ASSET", "EQ", "LNLSGR", "LNATRES", "NCLNLS", "ROA", "y")
        .withColumn("EQ_RATIO", F.col("EQ") / F.col("ASSET") * 100.0)
        .withColumn("NPL_RATIO", F.when(F.col("LNLSGR") != 0, F.col("NCLNLS") / F.col("LNLSGR") * 100.0))
        .withColumn("RES_RATIO", F.when(F.col("LNLSGR") != 0, F.col("LNATRES") / F.col("LNLSGR") * 100.0))
    )
    w_lag = Window.partitionBy("CERT").orderBy("QDATE")
    w_ma4 = Window.partitionBy("CERT").orderBy("QDATE").rowsBetween(-3, 0)

    df = (
        df.withColumn("ASSET_G4", F.col("ASSET") / F.lag("ASSET", 4).over(w_lag) - 1.0)
          .withColumn("EQ_RATIO_D4", F.col("EQ_RATIO") - F.lag("EQ_RATIO", 4).over(w_lag))
          .withColumn("ROA_MA4", F.when(F.count("ROA").over(w_ma4) == 4, F.avg("ROA").over(w_ma4)))
    )
    return (
        df.groupBy("QDATE")
          .agg(
              F.count("*").alias("n_rows"),
              F.median("EQ_RATIO").alias("med_EQ_RATIO"),
              F.median("NPL_RATIO").alias("med_NPL_RATIO"),
              F.median("ASSET_G4").alias("med_ASSET_G4"),
              F.median("ROA_MA4").alias("med_ROA_MA4"),
              F.sum("y").alias("n_positive"),
          )
          .orderBy("QDATE")
    )

q_spark = build_spark_query()

# 3. Xuat Execution Plan cua Catalyst Optimizer
buf = io.StringIO()
with redirect_stdout(buf):
    q_spark.explain(mode="formatted")
plan_text = buf.getvalue()
Path("outputs/part3_spark_plan.txt").write_text(plan_text, encoding="utf-8")

# 4. Do thoi gian PySpark vs Polars Lazy ngay tren cung may ao Colab (3 lan chay)
spark_times = []
for _ in range(3):
    t0 = time.perf_counter()
    res_spark = q_spark.toPandas()
    spark_times.append(time.perf_counter() - t0)

polars_times = []
for _ in range(3):
    t0 = time.perf_counter()
    _ = (
        pl.scan_parquet(PANEL)
        .filter(pl.col("QDATE") <= date(2009, 12, 31))
        .select("CERT", "QDATE", "ASSET", "EQ", "LNLSGR", "LNATRES", "NCLNLS", "ROA", "y")
        .with_columns(
            (pl.col("EQ") / pl.col("ASSET") * 100).alias("EQ_RATIO"),
            (pl.col("NCLNLS") / pl.when(pl.col("LNLSGR") != 0).then(pl.col("LNLSGR")) * 100).alias("NPL_RATIO"),
            (pl.col("LNATRES") / pl.when(pl.col("LNLSGR") != 0).then(pl.col("LNLSGR")) * 100).alias("RES_RATIO"),
        )
        .sort("CERT", "QDATE")
        .with_columns(
            (pl.col("ASSET") / pl.col("ASSET").shift(4).over("CERT") - 1).alias("ASSET_G4"),
            (pl.col("EQ_RATIO") - pl.col("EQ_RATIO").shift(4).over("CERT")).alias("EQ_RATIO_D4"),
            pl.col("ROA").rolling_mean(window_size=4).over("CERT").alias("ROA_MA4"),
        )
        .group_by("QDATE")
        .agg(
            pl.len().alias("n_rows"),
            pl.col("EQ_RATIO").median().alias("med_EQ_RATIO"),
            pl.col("NPL_RATIO").median().alias("med_NPL_RATIO"),
            pl.col("ASSET_G4").median().alias("med_ASSET_G4"),
            pl.col("ROA_MA4").median().alias("med_ROA_MA4"),
            pl.col("y").sum().alias("n_positive"),
        )
        .sort("QDATE")
        .collect()
    )
    polars_times.append(time.perf_counter() - t0)

# 5. Doi chieu sai so tuyet doi voi ket qua Polars tu laptop
ref = pd.read_parquet(REF_POLARS).sort_values("QDATE").reset_index(drop=True)
res_spark["QDATE"] = pd.to_datetime(res_spark["QDATE"])
ref["QDATE"] = pd.to_datetime(ref["QDATE"])
sp = res_spark.sort_values("QDATE").reset_index(drop=True)

assert len(sp) == len(ref), f"Row mismatch: {len(sp)} vs {len(ref)}"
diffs = {
    c: float((sp[c].astype(float) - ref[c].astype(float)).abs().max())
    for c in ref.columns if c != "QDATE"
}

report_lines = [
    f"Spark version: {spark.version} | Polars version: {pl.__version__}",
    f"SparkSession JVM startup: {t_init:.2f}s",
    f"Colab PySpark median (3 runs): { sorted(spark_times)[1]:.2f}s | runs: {[round(x, 2) for x in spark_times]}",
    f"Colab Polars Lazy median (3 runs): { sorted(polars_times)[1]:.2f}s | runs: {[round(x, 2) for x in polars_times]}",
    f"Quarters matched: {len(sp)} | Total dev rows: {int(sp['n_rows'].sum()):,} | Positives: {int(sp['n_positive'].sum()):,}",
    f"Max abs diff (PySpark vs Laptop Polars): {diffs}",
]
report_txt = "\n".join(report_lines)
print(report_txt)
print("\n--- SPARK FORMATTED PLAN (TOP 45 LINES) ---")
print("\n".join(plan_text.splitlines()[:45]))

Path("outputs/part3_spark_report.txt").write_text(report_txt + "\n\n" + plan_text, encoding="utf-8")

# Nen 2 file ket qua de tai ve may tinh 1 click
with zipfile.ZipFile("part3_spark_outputs.zip", "w") as zf:
    zf.write("outputs/part3_spark_plan.txt")
    zf.write("outputs/part3_spark_report.txt")
print("\nSaved part3_spark_outputs.zip - Tai file nay ve giai nen vao thu muc du an!")

Spark version: 4.0.5 | Polars version: 1.35.2
SparkSession JVM startup: 0.01s
Colab PySpark median (3 runs): 0.19s | runs: [35.08, 0.19, 0.18]
Colab Polars Lazy median (3 runs): 0.92s | runs: [0.92, 0.88, 1.09]
Quarters matched: 104 | Total dev rows: 1,295,349 | Positives: 9,751
Max abs diff (PySpark vs Laptop Polars): {'n_rows': 0.0, 'med_EQ_RATIO': 0.0, 'med_NPL_RATIO': 0.0, 'med_ASSET_G4': 0.0, 'med_ROA_MA4': 2.220446049250313e-16, 'n_positive': 0.0}

--- SPARK FORMATTED PLAN (TOP 45 LINES) ---
== Physical Plan ==
AdaptiveSparkPlan (13)
+- Sort (12)
   +- Exchange (11)
      +- ObjectHashAggregate (10)
         +- Exchange (9)
            +- ObjectHashAggregate (8)
               +- Project (7)
                  +- Window (6)
                     +- Sort (5)
                        +- Exchange (4)
                           +- Project (3)
                              +- Filter (2)
                                 +- Scan parquet  (1)


(1) Scan parquet 
Output [8]: [NCLNLS#12, ROA#

In [3]:
!pwd
!ls -la /content
!find / -name "panel_clean.parquet" 2>/dev/null

/content
total 20
drwxr-xr-x 1 root root 4096 Nov 12 01:00 .
drwxr-xr-x 1 root root 4096 Nov 12 00:55 ..
drwxr-xr-x 4 root root 4096 Oct 26 22:18 .config
drwxr-xr-x 2 root root 4096 Nov 12 01:00 outputs
drwxr-xr-x 1 root root 4096 Oct 26 22:18 sample_data
/panel_clean.parquet


In [4]:
!mv /panel_clean.parquet /content/
!mv /bench_polars_lazy.parquet /content/
!ls -la /content

total 167240
drwxr-xr-x 1 root root      4096 Nov 12 01:03 .
drwxr-xr-x 1 root root      4096 Nov 12 01:03 ..
-rw-r--r-- 1 root root     10478 Nov 12 00:55 bench_polars_lazy.parquet
drwxr-xr-x 4 root root      4096 Oct 26 22:18 .config
drwxr-xr-x 2 root root      4096 Nov 12 01:00 outputs
-rw-r--r-- 1 root root 171219352 Nov 12 00:58 panel_clean.parquet
drwxr-xr-x 1 root root      4096 Oct 26 22:18 sample_data


In [6]:
import os
print(os.getcwd())
!ls -la /content
from google.colab import files
files.download("/content/part3_spark_outputs.zip")

/content
total 167252
drwxr-xr-x 1 root root      4096 Nov 12 01:04 .
drwxr-xr-x 1 root root      4096 Nov 12 01:03 ..
-rw-r--r-- 1 root root     10478 Nov 12 00:55 bench_polars_lazy.parquet
drwxr-xr-x 4 root root      4096 Oct 26 22:18 .config
drwxr-xr-x 2 root root      4096 Nov 12 01:04 outputs
-rw-r--r-- 1 root root 171219352 Nov 12 00:58 panel_clean.parquet
-rw-r--r-- 1 root root     10093 Nov 12 01:04 part3_spark_outputs.zip
drwxr-xr-x 1 root root      4096 Oct 26 22:18 sample_data


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>